# 01 — Data Audit & Ingestion

Build the competition-season-match inventory, download raw data, and parse into canonical tables.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import yaml, pandas as pd
from src.data.ingest import get_competitions, get_matches
from src.data.inventory import build_inventory, get_360_match_ids
from src.data.parse_events import parse_events
from src.data.parse_360 import parse_360_frames, get_frame_summary
from src.data.parse_lineups import parse_lineups
from src.data.join_pass_frames import build_pass_instances

with open("../configs/data.yaml") as f:
    cfg = yaml.safe_load(f)


## 1. Competition inventory

In [ ]:
competitions = get_competitions()
print(f"Total competitions: {len(competitions)}")
competitions.head(10)


## 2. Match inventory with 360 coverage

In [ ]:
inventory = build_inventory(cfg["statsbomb"]["competitions"])
print(inventory.groupby(["competition_name", "has_360"])["match_id"].count().unstack(fill_value=0))
inventory.head()


## 3. Parse a sample match

In [ ]:
sample_match_id = inventory.loc[inventory["has_360"], "match_id"].iloc[0]
print(f"Sample match with 360: {sample_match_id}")

from src.data.ingest import get_events, get_360_frames
raw_events = get_events(sample_match_id)
raw_frames = get_360_frames(sample_match_id)

events = parse_events(raw_events)
frames = parse_360_frames(raw_frames)
frame_summary = get_frame_summary(frames)

print(f"Events: {len(events)}")
print(f"Frame rows: {len(frames)}")
print(f"Events with 360: {len(frame_summary)}")


## 4. Build pass_instances canonical table

In [ ]:
pass_instances = build_pass_instances(events, frame_summary)
print(f"Pass instances: {len(pass_instances)}")
print(f"With 360: {pass_instances['has_360'].sum()}")
print(f"\nColumns: {list(pass_instances.columns)}")
pass_instances.head()


## 5. Schema checks

In [ ]:
required_cols = ["event_uuid", "match_id", "start_x", "start_y", "end_x", "end_y",
                 "pass_length", "has_360", "n_visible_teammates", "n_visible_opponents"]
missing = [c for c in required_cols if c not in pass_instances.columns]
print("Missing required columns:", missing or "None — all present ✓")
print("\nMissingness report:")
print(pass_instances.isnull().mean().sort_values(ascending=False).head(15))
